In [1]:
import pandas as pd
import numpy as np

In [2]:

def clean_currency_and_numbers(df, columns):
    """Utility function to strip currency symbols, commas, and handle NaNs."""
    for col in columns:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(r'[₹,]', '', regex=True)
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)
    return df

def clean_string_columns(df, columns):
    """Utility function to trim whitespace and standardize casing."""
    for col in columns:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.upper()
    return df

In [3]:
def load_and_merge_mplads_datasets():
    print("[*] Loading all 6 MPLADS Datasets...")

    df1_allocated = pd.read_csv("assests/Allocated Limit for Honble MPs.csv")
    df2_calamity = pd.read_csv("assests/Amount consented for Calamity.csv")
    df3_recommended = pd.read_csv("assests/Works Recommended.csv")
    df4_sanctioned = pd.read_csv("assests/Works Sanctioned.csv")
    df5_completed = pd.read_csv("assests/Works Completed.csv")
    df6_expenditure = pd.read_csv("assests/Expenditure on Completed and On-going Works as on Date.csv")

    #simple cleaning of column names to remove leading/trailing spaces and standardize casing
    df1_allocated.columns = df1_allocated.columns.str.strip().str.title()
    df2_calamity.columns = df2_calamity.columns.str.strip().str.title()
    df3_recommended.columns = df3_recommended.columns.str.strip().str.title()
    df4_sanctioned.columns = df4_sanctioned.columns.str.strip().str.title()
    df5_completed.columns = df5_completed.columns.str.strip().str.title()
    df6_expenditure.columns = df6_expenditure.columns.str.strip().str.title()
    df1_allocated = df1_allocated.rename(columns={"Hon'Ble Members Of Parliaments": "Hon'Ble Members Of Parliament"})
    df6_expenditure['Work'] = df6_expenditure['Work Id'] + '-' + df6_expenditure['Work']

    print("[*] Standardizing and Cleaning Individual Datasets...")

    # =========================================================================
    # 2. CLEAN & AGGREGATE FINANCIALS (DATASETS 1 & 2)
    # Key: ['mp_id', 'financial_year']
    # =========================================================================
    df1_allocated = clean_currency_and_numbers(df1_allocated, ['Allocated Amount ( ₹ )'])
    df1_allocated = clean_string_columns(df1_allocated, ["Hon'Ble Members Of Parliament", 'Constituency'])

    df2_calamity = clean_currency_and_numbers(df2_calamity, ['Consent Amount ( ₹ )'])
    df2_calamity = clean_string_columns(df2_calamity, ["Hon'Ble Members Of Parliament"])

    # Aggregate calamity relief per MP per financial year
    calamity_agg = df2_calamity.groupby(["Hon'Ble Members Of Parliament"], as_index=False).agg(
        total_calamity_consented=('Consent Amount ( ₹ )', 'sum')
    )

    # Merge Allocated Funds with Calamity Consented
    mp_finances = pd.merge(
        df1_allocated, 
        calamity_agg, 
        on=["Hon'Ble Members Of Parliament"], 
        how='left'
    )
    mp_finances['total_calamity_consented'] = mp_finances['total_calamity_consented'].fillna(0.0)
    
    # Calculate Net Available Balance for the MP
    mp_finances['net_available_fund'] = mp_finances['Allocated Amount ( ₹ )'] - mp_finances['total_calamity_consented']

    # =========================================================================
    # 3. CLEAN WORK LIFECYCLE DATASETS (DATASETS 3, 4, 5 & 6)
    # Key: ['work_id']
    # =========================================================================
    # Dataset 3: Works Recommended
    df3_recommended = clean_currency_and_numbers(df3_recommended, ['Recommended Amount   ( ₹ )'])
    df3_recommended = clean_string_columns(df3_recommended, ['Work', "Hon'Ble Members Of Parliament", 'Work Description', 'Constituency', 'State','Work Category'])
    df3_recommended['Recommended Date'] = pd.to_datetime(df3_recommended['Recommended Date'], errors='coerce')
    df3_recommended['Sanction Date'] = pd.to_datetime(df3_recommended['Sanction Date'], errors='coerce')
    
    # Dataset 4: Works Sanctioned
    df4_sanctioned = clean_currency_and_numbers(df4_sanctioned, ['Sanction Amount ( ₹ )'])
    df4_sanctioned = clean_string_columns(df4_sanctioned, ['Work', 'Ida', "Hon'Ble Members Of Parliament", 'Work Description', 'Vendor Name', 'Constituency', 'State','Work Category', 'Work Status'])
    df4_sanctioned['Recommended Date'] = pd.to_datetime(df4_sanctioned['Recommended Date'], errors='coerce')
    df4_sanctioned['Sanction Date'] = pd.to_datetime(df4_sanctioned['Sanction Date'], errors='coerce')
    
    # Dataset 5: Works Completed
    df5_completed = clean_string_columns(df5_completed, ['Work', "Hon'Ble Members Of Parliament", 'Work Description', 'Constituency', 'State','Work Category'])
    df5_completed['Completion Date'] = pd.to_datetime(df5_completed['Completion Date'], errors='coerce')
    
    # Dataset 6: Expenditure on Completed and Ongoing Works
    df6_expenditure = clean_currency_and_numbers(df6_expenditure, ['Fund Disbursed Amount ( ₹ )'])
    df6_expenditure = clean_string_columns(df6_expenditure, ['State','Work', "Hon'Ble Members Of Parliament", 'Vendor Name', 'Payment Status'])
    df6_expenditure['Expenditure Date'] = pd.to_datetime(df6_expenditure['Expenditure Date'], errors='coerce')
    
    # Group Dataset 6 by work_id in case there are multiple payment installments
    expenditure_agg = df6_expenditure.groupby('Work', as_index=False).agg(
        total_expenditure_released=('Fund Disbursed Amount ( ₹ )', 'max'),
        vendor_names=('Vendor Name', lambda x: ', '.join(x.dropna().unique())),
        latest_payment_date=('Expenditure Date', 'max'),
        payment_status=('Payment Status', 'last')
        )
    
    print("[*] Joining Datasets into Work-Level Master Dataframe...")

    # =========================================================================
    # 4. JOIN WORK-LEVEL DATASETS (3 -> 4 -> 5 -> 6)
    # =========================================================================
    # Step A: Recommended + Sanctioned
    works_master = pd.merge(
            df3_recommended, 
            df4_sanctioned, 
            on='Work', 
            how='left',
            suffixes=('', '_sanctioned')
        )
    
    # Step B: + Completed
    works_master = pd.merge(
            works_master, 
            df5_completed[['Work', 'Completion Date', 'Image']], 
            on='Work', 
            how='left'
        )
    
    # Step C: + Expenditure
    works_master = pd.merge(
            works_master, 
            expenditure_agg, 
            on='Work', 
            how='left'
        )

    # =========================================================================
    # 5. JOIN WITH FINANCIAL ALLOCATIONS (DATASETS 1 & 2)
    # =========================================================================
    master_df = pd.merge(
            works_master, 
            mp_finances[["Hon'Ble Members Of Parliament", 'Allocated Amount ( ₹ )', 'total_calamity_consented', 'net_available_fund']], 
            on=["Hon'Ble Members Of Parliament"], 
            how='left'
        )
    
    print("[*] Calculating Derived Audit Indicators & Discrepancy Flags...")

    # =========================================================================
    # 6. FEATURE ENGINEERING & ANOMALY INDICATORS
    # =========================================================================
    # A. Status Tracking
    master_df['is_sanctioned'] = master_df['Sanction Date'].notna()
    master_df['is_completed'] = master_df['Completion Date'].notna()
    master_df['has_expenditure'] = master_df['total_expenditure_released'] > 0
    
    # B. Cost Variance (Sanctioned vs Expenditure)
    master_df['cost_variance'] = master_df['total_expenditure_released'] - master_df['Sanction Amount ( ₹ )']
    master_df['is_cost_overrun'] = master_df['cost_variance'] > 0
    
    # C. Days Taken To Sanction (Recommendation to Sanction)
    master_df['days_to_sanction'] = (master_df['Sanction Date'] - master_df['Recommended Date']).dt.days
    
    # D. Days Taken To Complete (Sanction to Completion)
    master_df['days_to_complete'] = (master_df['Completion Date'] - master_df['Sanction Date']).dt.days

    # E. Discrepancy Flag: Payment made on unsanctioned work
    master_df['flag_unsanctioned_payment'] = (~master_df['is_sanctioned']) & (master_df['has_expenditure'])
    
    # F. Discrepancy Flag: Marked completed without image
    master_df['flag_missing_completion_cert'] = (master_df['is_completed']) & (master_df['Image'].isna() | (master_df['Image'].str.strip() == ''))
    
    print("[+] Master Dataframe built successfully!")
    print(f"    • Total Master Records: {len(master_df):,}")
    print(f"    • Total Unique MPs: {master_df["Hon'Ble Members Of Parliament"].nunique()}")
    print(f"    • Total Unique Works: {master_df['Work'].nunique()}")
    
    return master_df

In [4]:
if __name__ == "__main__":
    # Run pipeline
    master_df = load_and_merge_mplads_datasets()
    master_df.to_csv("assests/mplads_master_analytics_ready.csv", index=False)
    pass

[*] Loading all 6 MPLADS Datasets...


C:\Users\goure\AppData\Local\Temp\ipykernel_7656\1949271756.py:6: DtypeWarning: Columns (0: Sr. No., 1: RECOMMENDED AMOUNT   ( ₹ )) have mixed types. Specify dtype option on import or set low_memory=False.
  df3_recommended = pd.read_csv("assests/Works Recommended.csv")
C:\Users\goure\AppData\Local\Temp\ipykernel_7656\1949271756.py:7: DtypeWarning: Columns (0: Sr. No., 1: Sanction Amount ( ₹ )) have mixed types. Specify dtype option on import or set low_memory=False.
  df4_sanctioned = pd.read_csv("assests/Works Sanctioned.csv")
C:\Users\goure\AppData\Local\Temp\ipykernel_7656\1949271756.py:9: DtypeWarning: Columns (0: Sr. No., 1: Fund Disbursed Amount ( ₹ )) have mixed types. Specify dtype option on import or set low_memory=False.
  df6_expenditure = pd.read_csv("assests/Expenditure on Completed and On-going Works as on Date.csv")


[*] Standardizing and Cleaning Individual Datasets...
[*] Joining Datasets into Work-Level Master Dataframe...
[*] Calculating Derived Audit Indicators & Discrepancy Flags...
[+] Master Dataframe built successfully!
    • Total Master Records: 107,090
    • Total Unique MPs: 539
    • Total Unique Works: 78962
